# 02 · Baselines and controlled extensions

M0 predicts the training-set majority class. M1 uses fixed-priority rules. M2 uses word TF-IDF + LinearSVC, and M3 combines word and character TF-IDF + LinearSVC. Only C is tuned, using the candidates {0.1, 1, 10} and development Macro-F1. Ties select the smaller C. Vocabulary, IDF, and the classifier are fitted only on training data.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "config.json").exists():
    ROOT = ROOT.parent
assert (ROOT / "config.json").exists(), "Open this notebook inside the project."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)
import pandas as pd


In [2]:
from src.train import train
dev_metrics = train()
dev_metrics

Matplotlib is building the font cache; this may take a moment.


{"model_id": "M0", "C": null, "train_n": 14158, "n": 1880, "accuracy": 0.28882978723404257, "macro_f1": 0.07470078415187784}


{"model_id": "M1", "C": null, "train_n": 14158, "n": 1880, "accuracy": 0.9893617021276596, "macro_f1": 0.9862074708977498}


{"model_id": "M2", "C": 1.0, "train_n": 14158, "n": 1880, "accuracy": 0.999468085106383, "macro_f1": 0.9992078205648841}


{"model_id": "M3", "C": 0.1, "train_n": 14158, "n": 1880, "accuracy": 0.999468085106383, "macro_f1": 0.9992078205648841}


,model_id,C,train_n,n,accuracy,macro_f1
0,M0,NaN,14158,1880,0.288830,0.074701
1,M1,NaN,14158,1880,0.989362,0.986207
2,M2,1.0,14158,1880,0.999468,0.999208
3,M3,0.1,14158,1880,0.999468,0.999208


In [3]:
pd.read_csv('results/dev/tuning.csv')

,model_id,C,n,accuracy,macro_f1
0,M0,NaN,1880,0.288830,0.074701
1,M1,NaN,1880,0.989362,0.986207
2,M2,0.1,1880,0.998936,0.998660
3,M2,1.0,1880,0.999468,0.999208
4,M2,10.0,1880,0.999468,0.999208
5,M3,0.1,1880,0.999468,0.999208
6,M3,1.0,1880,0.999468,0.999208
7,M3,10.0,1880,0.999468,0.999208


## Human materials required for M4/M5

The worksheets contain 180 training parent questions and 120 test parent questions, selected after splitting without consulting predictions. Training and test paraphrases cannot share parents. Manually fill in each paraphrase's text, author, reviewer, and review_notes; mark it approved only after a real review. Spelling variants are generated from the original question, independently of paraphrases, using fixed rules, and also require review.

The following cell creates blank review worksheets. It does not write or approve paraphrases.

In [4]:
from src.variants import make_worksheets, reviewed_variants
make_worksheets()
worksheet = pd.read_csv('data/paraphrases/train_paraphrases.csv', keep_default_na=False)
print(worksheet.review_status.value_counts())
worksheet.head(3)

Preserved existing human worksheet: train_paraphrases.csv
Preserved existing human worksheet: test_variants.csv
review_status
pending    180
Name: count, dtype: int64


,variant_id,parent_question_id,variant_type,original_text,text,label,disease_group,split,review_status,author,reviewer,review_notes,change_record
0,train::8_NHLBI_QA_XML/0000119.xml::0000119-1::...,8_NHLBI_QA_XML/0000119.xml::0000119-1,paraphrase,What is (are) Sleep Deprivation and Deficiency ?,,Basic information,sleep deprivation and deficiency,train,pending,,,,
1,train::2_GARD_QA/0002400.xml::0002400-1::parap...,2_GARD_QA/0002400.xml::0002400-1,paraphrase,What is (are) Focal dystonia ?,,Basic information,focal dystonia,train,pending,,,,
2,train::2_GARD_QA/0005625.xml::0005625-1::parap...,2_GARD_QA/0005625.xml::0005625-1,paraphrase,What is (are) Sideroblastic anemia pyridoxine-...,,Basic information,sideroblastic anemia pyridoxine refractory aut...,train,pending,,,,


In [5]:
approved = reviewed_variants('data/paraphrases/train_paraphrases.csv', 'train')
if approved.empty:
    print('M4/M5 pending: no human-approved training paraphrases.')
else:
    print('Reviewed candidates:', len(approved))
    print('After all reviews, explicitly train all six together:')
    print('python -m src.train --models M0 M1 M2 M3 M4 M5')

M4/M5 pending: no human-approved training paraphrases.


The development and test splits remain fixed. Do not use final test scores to revise rules or choose parameters. Near-perfect scores may reflect repeated question templates; the independent expression-variation experiment remains necessary.